# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [5]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

import re
import logging

# Basic logging setup (Bonus: Add logging)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("single_agent")


def agent(query: str):
    """
    Single-Agent Smart Assistant.
    Routes the incoming query to the right tool based on intent keywords,
    and always returns a structured JSON-style dict:
        {"type": "calculation / keywords / general / error", "result": ...}
    """
    try:
        if not isinstance(query, str) or not query.strip():
            logger.warning("Empty or invalid query received.")
            return {"type": "error", "result": "Query cannot be empty."}

        query_lower = query.lower()
        logger.info(f"Received query: {query!r}")

        # ---- Route 1: Math / Calculation ----
        if "calculate" in query_lower:
            match = re.search(r"calculate\s+(.*)", query, flags=re.IGNORECASE)
            expression = match.group(1).strip() if match else ""

            if not expression:
                logger.error("No expression found after 'calculate'.")
                return {"type": "error", "result": "No expression provided to calculate."}

            if not re.fullmatch(r"[0-9\.\+\-\*\/\(\)\s%]+", expression):
                logger.error(f"Unsafe/invalid expression blocked: {expression!r}")
                return {"type": "error", "result": "Invalid characters in expression."}

            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                logger.error(f"Calculator failed on expression: {expression!r}")
                return {"type": "error", "result": calc_result}

            logger.info(f"Calculation successful: {expression} = {calc_result}")
            return {"type": "calculation", "result": calc_result}

        # ---- Route 2: Keyword extraction ----
        elif "keywords" in query_lower:
            match = re.search(r"keywords\s+from\s+(.*)", query, flags=re.IGNORECASE)
            text_to_process = match.group(1).strip() if match else query

            if not text_to_process:
                logger.error("No text provided for keyword extraction.")
                return {"type": "error", "result": "No text provided to extract keywords from."}

            keywords = extract_keywords(text_to_process)
            logger.info(f"Extracted keywords: {keywords}")
            return {"type": "keywords", "result": keywords}

        # ---- Route 3: General / fallback ----
        else:
            logger.info("Routed to general response handler.")
            return {
                "type": "general",
                "result": f"I received your query: '{query}'. I don't have a specific tool for this, "
                           f"but feel free to ask me to 'calculate ...' or extract 'keywords from ...'."
            }

    except Exception as e:
        logger.exception("Unexpected error while processing the agent request.")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [6]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['intelligence', 'artificial', 'industries', 'transforming']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. I don't have a specific tool for this, but feel free to ask me to 'calculate ...' or extract 'keywords from ...'."}
--------------------------------------------------


In [7]:
def count_words(text: str) -> int:
    """Count number of words in the given text."""
    try:
        return len(text.split())
    except Exception:
        return 0


def agent_v2(query: str):
    """Extended router that also supports the word-count tool."""
    query_lower = query.lower()
    try:
        if "count words" in query_lower:
            match = re.search(r"count words\s+in\s+(.*)", query, flags=re.IGNORECASE)
            text_to_process = match.group(1).strip() if match else query
            if not text_to_process:
                return {"type": "error", "result": "No text provided to count words."}
            logger.info(f"Counting words in: {text_to_process!r}")
            return {"type": "word_count", "result": count_words(text_to_process)}
        # fall back to the original agent for everything else
        return agent(query)
    except Exception as e:
        logger.exception("Unexpected error in agent_v2.")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}


# quick test of the bonus tool
print(agent_v2("count words in Autogen makes building agents easy"))
print(agent_v2("Calculate 15 * 3"))


{'type': 'word_count', 'result': 5}
{'type': 'calculation', 'result': '45'}


In [10]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): Calculate 30*2
Response: {'type': 'calculation', 'result': '60'}
Enter query (type 'exit' to stop): Extract keywords from I am a CELEBAL INTERN
Response: {'type': 'keywords', 'result': ['celebal', 'intern']}
Enter query (type 'exit' to stop): exit
